# Etapa 3 — Rodando no KAGGLE (GPU 16 GB, ~30 GB RAM)

**Antes de rodar, configure no painel da direita (`Settings`/`...`):**
1. **Accelerator → GPU** (P100 ou T4)
2. **Internet → On** (necessário p/ baixar os pesos pré-treinados e clonar o repo)
3. **Add Input → Datasets** → adicione o dataset que contém o `pathmnist_224.npz` (veja a célula 1 sobre como subir esse arquivo).

## 0. (Uma vez) Subir o dataset pro Kaggle
O Kaggle não rebaixa o dataset a cada sessão se você o transformar num *Kaggle Dataset*:
1. Baixe o `pathmnist_224.npz` pelo navegador: https://zenodo.org/records/10519652/files/pathmnist_224.npz?download=1
2. No Kaggle: `Create → New Dataset` → suba o arquivo (mantenha o nome `pathmnist_224.npz`).
3. Depois, neste notebook, use `Add Input` para anexá-lo. Ele aparece em `/kaggle/input/...`

## 1. Conferir GPU

In [ ]:
import torch
print('CUDA?', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Trazer o código do repositório
Troque pela URL do repositório de vocês (Internet precisa estar On).

In [ ]:
import os, sys
REPO_URL = 'https://github.com/SEU_USUARIO/trabalho-final-ia.git'  # <-- EDITE
%cd /kaggle/working
![ -d trabalho-final-ia] || git clone $REPO_URL
%cd trabalho-final-ia
ROOT = os.getcwd()
if ROOT not in sys.path: sys.path.insert(0, ROOT)
print('raiz:', ROOT)

## 3. Instalar só o que falta
O Kaggle já vem com PyTorch+CUDA. NÃO reinstale o torch (quebraria o CUDA). Só o medmnist é necessário aqui.

In [ ]:
!pip install -q medmnist

## 4. Localizar o dataset anexado
Acha o `.npz` automaticamente dentro de `/kaggle/input`.

In [ ]:
import glob
hits = glob.glob('/kaggle/input/**/pathmnist_224.npz', recursive=True)
assert hits, 'Anexe o dataset com pathmnist_224.npz via Add Input.'
DATA_ROOT = os.path.dirname(hits[0])
print('dataset em:', DATA_ROOT)

## 5. Montar os DataLoaders 224×224
Com 30 GB de RAM dá pra carregar nativo. `batch_size=64` cabe nos 16 GB de VRAM.

In [ ]:
from src.utils.seeds import set_seed
from src.etapa2_pytorch.data_224 import make_loaders
set_seed(42)
train_loader, val_loader, test_loader = make_loaders(
    size=224, batch_size=64, num_workers=2, augment=True, root=DATA_ROOT)
print('batches treino:', len(train_loader))

## 6. Varredura otimizador × learning rate
No ResNet50 (fine-tuning) — agora viável, com 16 GB de VRAM.

In [ ]:
from src.etapa3_cnn_vit.run_experiments import sweep_optim_lr
melhor = sweep_optim_lr('resnet50', 'fine_tuning',
                        train_loader, val_loader, epochs=5, pretrained=True)
print('Melhor:', melhor)

## 7. Comparação de todos os modelos
CNN autoral + 3 CNNs (feature extraction e fine-tuning) + ViT. Gera `results/logs/etapa3_comparacao_modelos.csv` e os checkpoints.

⚠️ Sessão do Kaggle dura ~9h (GPU) e desliga por inatividade — não feche a aba.

In [ ]:
from src.etapa3_cnn_vit.run_experiments import compare_models
resultados = compare_models(train_loader, val_loader,
                            optimizer=melhor['otimizador'], lr=melhor['lr'],
                            epochs=8, pretrained=True)
import pandas as pd; pd.DataFrame(resultados)

## 8. Salvar os resultados antes de fechar
O que está em `/kaggle/working` vira *Output* do notebook quando você faz `Save Version`. Baixe os CSVs e os checkpoints de `results/` para juntar ao relatório/repositório.

In [ ]:
!ls -la results/logs results/checkpoints

## 9. (Só no fim do projeto) Avaliação no TESTE
O conjunto de teste só pode ser usado **uma vez**, sobre o melhor modelo final (após a Etapa 5).